# 02 — Validasi anotasi

Memeriksa: struktur COCO (id, `category_id`, koordinat piksel), pemetaan kelas eksplisit, gambar yang dilewati
`prepare` (dan alasannya), label YOLO hilang/tidak valid, berkas duplikat (byte identik & hampir identik),
serta kebocoran split.

Gambar tanpa anotasi **tidak** dianggap contoh negatif kecuali dokumentasi dataset menyatakannya
(`empty_images_are_negative` di config prepare).

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## 1. COCO mentah

In [ ]:
import glob
from wallacea_ml.coco import load_coco, validate_coco
rows = []
for p in sorted(glob.glob(str(ML_ROOT / "data/raw/uavmulticrop/UAVmulticrop/maize/*.json"))):
    r = validate_coco(load_coco(p))
    s = r.summary()
    rows.append({"file": Path(p).name, "images": s["images"], "boxes": s["annotations"], "categories": s["categories"],
                 "unannotated": s["unannotated_images"], "errors": dict(s["errors"]), "warnings": dict(s["warnings"])})
display(_df(rows))

## 2. Pemetaan kelas (harus eksplisit; kategori tak terpetakan = error)

In [ ]:
import yaml
from wallacea_ml.coco import resolve_class_map
cmap = yaml.safe_load((ML_ROOT / "configs/class_maps/uavmulticrop.yaml").read_text(encoding="utf-8"))
print("class_map:", cmap)
print("category_id → label:", resolve_class_map(load_coco(p), cmap))

## 3. Laporan `prepare` — gambar dilewati & statistik konversi

In [ ]:
DATASET = "uavmulticrop_maize"
ROOT = ML_ROOT / "data" / "prepared" / DATASET
rep = json.loads((ROOT / "prepare_report.json").read_text(encoding="utf-8"))
print("ditulis:", rep["images_written"], "gambar,", rep["boxes_written"], "box")
print("konversi:", rep["conversion"])
print("dilewati per alasan:", rep["skipped_by_reason"])
display(_df(rep["skipped"][:50]))

## 4. Label YOLO hasil prepare + duplikat

In [ ]:
from wallacea_ml.manifest import load_manifest
from wallacea_ml.inspection import inspect_manifest, find_duplicates
m = load_manifest(ROOT / "manifest.json")
st = inspect_manifest(m, ROOT)
print("isu label:", st.summary()["issues"] or "tidak ada")
display(_df([vars(i) for i in st.issues[:50]]))
dups = find_duplicates(m, ROOT)
print("byte identik:", dups["exact"] or "tidak ada")
print("hampir identik:", len(dups["near"]), "pasang;", sum(d["cross_group"] for d in dups["near"]), "lintas grup (calon kebocoran)")
display(_df(dups["near"][:30]))

## 5. Box mencurigakan (sangat kecil / menyentuh tepi)

In [ ]:
from wallacea_ml import backend
from wallacea_ml.viz import draw_boxes
tiny = sorted(st.boxes, key=lambda b: b["w_px"] * b["h_px"])[:5]
print([(b["image"], round(b["w_px"], 1), round(b["h_px"], 1)) for b in tiny])
if tiny:
    e = tiny[0]
    img = backend.load_oriented(ROOT / e["image"])
    plt.figure(figsize=(8, 5)); plt.imshow(draw_boxes(img, [("check", [e["l"], e["t"], e["w"], e["h"]])])); plt.axis("off"); plt.show()

## 6. Split & kebocoran

In [ ]:
SPLIT_DIR = ML_ROOT / "data" / "splits" / DATASET
rep = json.loads((SPLIT_DIR / "split_report.json").read_text(encoding="utf-8"))
print(json.dumps(rep, indent=1))
assert not rep["leaks"], "Split bocor — jangan training/evaluasi sebelum diperbaiki"
